# Data Mining — Lab 2 (Version 2)
**Adventist University of Central Africa — MSc in Big Data Analytics**

**Dataset:** Credit Approval dataset (UCI / `crx.data`)

This is an alternative preprocessing approach to compare against Version 1:

| Step | Version 1 | Version 2 (this notebook) |
|---|---|---|
| Missing values | Impute (median / mode) | Drop rows with missing values |
| Outliers | IQR capping | Z-score removal |
| Categorical encoding | Label Encoding | One-Hot Encoding |
| Scaling | StandardScaler | MinMaxScaler |

This notebook covers the same three stages:
1. Data Preparation
2. Exploratory Data Analysis (EDA)
3. Preprocessing, Feature Selection and Engineering


## 1. Data Preparation

Import the libraries needed for this lab.

In [ ]:
from sklearn.metrics import roc_auc_score
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


The raw `crx.data` file has no header row, so the columns are renamed to
`A1`–`A16` (A16 is the target: `+` = approved, `-` = not approved).

In [ ]:
cols = [f"A{i}" for i in range(1, 17)]
Demo = pd.read_csv("crx.data", header=None, names=cols)


Preview the data and confirm the columns were renamed correctly.

In [ ]:
Demo


In [ ]:
Demo.columns


This dataset uses `?` as its missing-value marker instead of a blank cell.
We replace it with `NaN` so pandas can recognize it. Columns A2 and A14 are
numeric but were read as text because of the `?` values, so we convert them
back to numbers.

In [ ]:
# "?" is this dataset's missing value marker -> replace with NaN
Demo.replace('?', np.nan, inplace=True)

# A2 and A14 are numeric but got read as text because of the "?" strings
Demo['A2'] = pd.to_numeric(Demo['A2'])
Demo['A14'] = pd.to_numeric(Demo['A14'])


Save the prepared dataframe to a CSV file, as required by the lab.

In [ ]:
Demo.to_csv("credit_approval_prepared_v2.csv", index=False)


## 2. Exploratory Data Analysis (EDA)

We check for missing values, look at summary statistics, and visualize
distributions and outliers.

Check how many missing values are in each column.

In [ ]:
Demo.isnull().sum()


Summary statistics for the numeric columns.

In [ ]:
Demo.describe()


Histograms to see how each numeric column is distributed.

In [ ]:
Demo.hist(figsize=(12, 8))
plt.show()


Boxplots to visually identify outliers in the numeric columns.

In [ ]:
Demo.select_dtypes(include='number').boxplot(figsize=(12, 6))
plt.show()


## 3. Preprocessing, Feature Selection and Engineering

This is where Version 2 differs from Version 1. Instead of imputing
missing values and capping outliers, we **drop** affected rows and use
**Z-score** outlier removal. Categorical columns are **one-hot encoded**
instead of label encoded, and numeric columns are scaled with
**MinMaxScaler** instead of StandardScaler.

Separate the numeric and categorical columns (excluding the target, A16).

In [ ]:
num_cols = Demo.select_dtypes(include='number').columns
cat_cols = Demo.select_dtypes(include='object').columns.drop('A16')


**Missing values:** drop any row that has a missing value, rather than imputing it.

In [ ]:
# drop rows with any missing values
Demo = Demo.dropna().reset_index(drop=True)


**Outliers:** remove rows where a numeric value is more than 3 standard deviations from the mean (Z-score method).

In [ ]:
from scipy import stats

# keep only rows where all numeric columns have |z-score| < 3
z_scores = np.abs(stats.zscore(Demo[num_cols]))
Demo = Demo[(z_scores < 3).all(axis=1)].reset_index(drop=True)


**Categorical encoding:** one-hot encode the categorical columns (each category
becomes its own 0/1 column), instead of label encoding. The target (A16) is
still label encoded since it's binary.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# one-hot encode categorical feature columns
Demo = pd.get_dummies(Demo, columns=cat_cols, drop_first=True)

# label encode the binary target column
le = LabelEncoder()
Demo['A16'] = le.fit_transform(Demo['A16'])


**Scaling:** scale numeric columns to a 0–1 range using MinMaxScaler, instead of standardizing them.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# scale numeric features to a 0-1 range
scaler = MinMaxScaler()
Demo[num_cols] = scaler.fit_transform(Demo[num_cols])


Preview the fully prepared dataset, ready for model building in the next step.

In [ ]:
Demo.head()


## 4. Model Creation and Evaluation

### 4a. Classification Model

We split the data into train/test sets, train a Logistic Regression
classifier, and evaluate it with two metrics: Accuracy and ROC-AUC.

Separate features (X) and target (y), then split into train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

# separate features and target
X = Demo.drop(columns=['A16'])
y = Demo['A16']

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


Train a Logistic Regression model on the training set.

In [ ]:
from sklearn.linear_model import LogisticRegression

# train a Logistic Regression classifier
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]


**Metric 1 — Accuracy:** overall proportion of correct predictions.

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)


**Metric 2 — ROC-AUC:** measures ranking quality regardless of the
classification threshold, and is more robust to class imbalance.

In [ ]:
auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", auc)


**Why these two metrics:** Accuracy gives a simple overall correctness
rate, but credit approval data is often imbalanced (more approvals or more
rejections). ROC-AUC is threshold-independent and measures how well the
model ranks positive vs. negative cases, making it more reliable when
classes aren't perfectly balanced. Using both together gives a simple
headline number (accuracy) alongside a more robust check (ROC-AUC).